# Módulo 4 · Deep Learning Generativo y Auto-supervisado

## Objetivos
1. Entender el **aprendizaje de representaciones** con autoencoders y su versión probabilística, el **VAE** (ELBO, truco de reparametrización).
2. Implementar una **GAN** mínima y comprender por qué su entrenamiento es inestable (juego minimax).
3. Implementar un **modelo de difusión (DDPM)** desde cero sobre datos 2D y ver la conexión con el *score matching*.
4. Implementar **Flow Matching**, el paradigma que en 2024-2026 sustituyó a la difusión clásica en los generadores de imagen/vídeo punteros (Stable Diffusion 3, FLUX, Sora-like), y compararlo con DDPM.
5. Entender el **aprendizaje auto-supervisado** (SimCLR, DINO, MAE) que produce los *backbones* visuales de propósito general.
6. Saber usar el ecosistema práctico: **Hugging Face `diffusers`**, LoRA para personalizar modelos, y despliegue en nube.

## Mapa de lectura (capítulos verificados)
| Tema | Libro | Capítulo verificado |
|---|---|---|
| Autoencoders (apilados, convolucionales, *denoising*, dispersos), VAE, GAN, **difusión** | Géron, *Hands-On ML* (3.ª ed.) | **Cap. 17** Autoencoders, GANs, and Diffusion Models |
| GAN y DCGAN con código | Zhang et al., *Dive into Deep Learning* | **Cap. 20** Generative Adversarial Networks |
| Modelos multimodales y control de Stable Diffusion | Fregly, Barth & Eigenbrode, *Generative AI on AWS* | **Cap. 10** Multimodal Foundation Models · **Cap. 11** Controlled Generation and Fine-Tuning with Stable Diffusion |
| Cuantización y cómputo distribuido para modelos generativos | Fregly et al. | **Cap. 4** Quantization and Distributed Computing |

> **Nota de actualización (2026).** La malla original citaba "Zhang et al. Cap. 7 (Autoencoders) y Cap. 8 (GANs)": en D2L el Cap. 7-8 son **CNN**; las GAN están en el **Cap. 20** y D2L no tiene capítulo de autoencoders (ese contenido está en Géron Cap. 17). Se añade **Flow Matching** y el aprendizaje **auto-supervisado**, que no aparecen en los libros base pero son la práctica dominante hoy; se citan los artículos primarios.

> **Ruta de aprendizaje Intensiva en ML Aplicado y DL Moderno** · Notebook del **Módulo 4 · Deep Learning Generativo y Auto-supervisado**
> Licencia CC BY-NC-SA 4.0 · © 2025-2026 Alexander Saravia
>
> Cómo usar este notebook: léelo de arriba hacia abajo. Cada sección tiene (1) la **idea** explicada con palabras y fórmulas,
> (2) **código ejecutable** que la materializa desde cero o con la librería estándar de 2026, (3) **lectura guiada** con
> el capítulo *verificado* de la bibliografía y (4) **ejercicios**. Las celdas marcadas `# OPCIONAL` requieren dependencias
> pesadas (GPU, Hugging Face, Spark) y están protegidas con `try/except` para que el resto del notebook siga corriendo.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, time
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    TORCH = True; torch.manual_seed(0); device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("PyTorch", torch.__version__, device)
except ImportError:
    TORCH = False; print("PyTorch no instalado; este módulo lo requiere (`uv pip install torch`).")
rng = np.random.default_rng(0)

## 1. Autoencoders: comprimir para entender (Géron 17.1-17.6)

Un autoencoder aprende $\mathbf{z} = E(\mathbf{x})\in\mathbb{R}^k$ con $k\ll d$ y una reconstrucción $D(\mathbf{z})\approx\mathbf{x}$ minimizando $\|\mathbf{x}-D(E(\mathbf{x}))\|^2$.
* Con $E, D$ **lineales** y pérdida cuadrática, el óptimo es exactamente **PCA** (Géron 17.2): el subespacio que aprende es el de las $k$ primeras componentes.
* Con no linealidades obtiene variedades curvas. Variantes: *denoising* (entrada con ruido → salida limpia), *sparse* (penalizar activaciones), convolucional.

Comprobamos la equivalencia lineal-PCA sobre `digits`.

In [ ]:
if TORCH:
    from sklearn.datasets import load_digits
    from sklearn.decomposition import PCA
    d = load_digits(); X = torch.tensor(d.data / 16.0, dtype=torch.float32); y = d.target
    Xc = X - X.mean(0)

    k = 2
    lin_ae = nn.Sequential(nn.Linear(64, k, bias=False), nn.Linear(k, 64, bias=False))
    opt = torch.optim.Adam(lin_ae.parameters(), lr=1e-2)
    for it in range(1500):
        opt.zero_grad(); loss = F.mse_loss(lin_ae(Xc), Xc); loss.backward(); opt.step()
    pca = PCA(k).fit(Xc.numpy()); err_pca = ((pca.inverse_transform(pca.transform(Xc.numpy())) - Xc.numpy()) ** 2).mean()
    print(f"error reconstrucción  AE lineal = {loss.item():.5f}   PCA(k={k}) = {err_pca:.5f}  -> mismo subespacio óptimo")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    Z_ae = lin_ae[0](Xc).detach().numpy(); Z_pca = pca.transform(Xc.numpy())
    axes[0].scatter(*Z_ae.T, c=y, cmap="tab10", s=6); axes[0].set_title("código del AE lineal (2D)")
    axes[1].scatter(*Z_pca.T, c=y, cmap="tab10", s=6); axes[1].set_title("PCA (2D): misma geometría, rotada/escala"); plt.show()

## 2. Autoencoder variacional (VAE) (Géron 17.7)

Queremos un **modelo generativo**: muestrear $\mathbf{z}\sim p(\mathbf{z})=\mathcal N(0,I)$ y decodificar. El VAE (Kingma & Welling, 2014) hace que el codificador produzca una *distribución* $q_\phi(\mathbf{z}|\mathbf{x})=\mathcal N(\boldsymbol\mu,\boldsymbol\sigma^2)$ y maximiza la cota inferior de la evidencia (**ELBO**):

$$\log p(\mathbf{x}) \ \ge\ \underbrace{\mathbb{E}_{q_\phi}\big[\log p_\theta(\mathbf{x}|\mathbf{z})\big]}_{\text{reconstrucción}} - \underbrace{\mathrm{KL}\big(q_\phi(\mathbf{z}|\mathbf{x})\,\|\,\mathcal N(0,I)\big)}_{\text{regularización del espacio latente}}.$$

Para gaussianas diagonales la KL es cerrada: $\tfrac12\sum_j(\mu_j^2+\sigma_j^2-\log\sigma_j^2-1)$. El **truco de reparametrización** $\mathbf{z}=\boldsymbol\mu+\boldsymbol\sigma\odot\boldsymbol\epsilon$ permite retropropagar a través del muestreo.

In [ ]:
if TORCH:
    class VAE(nn.Module):
        def __init__(self, d=64, h=128, k=2):
            super().__init__()
            self.enc = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU())
            self.mu, self.logvar = nn.Linear(h, k), nn.Linear(h, k)
            self.dec = nn.Sequential(nn.Linear(k, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, d), nn.Sigmoid())
        def encode(self, x):
            hh = self.enc(x); return self.mu(hh), self.logvar(hh)
        def forward(self, x):
            mu, logvar = self.encode(x)
            z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)       # reparametrización
            return self.dec(z), mu, logvar

    def elbo_loss(x_hat, x, mu, logvar, beta=1.0):
        rec = F.binary_cross_entropy(x_hat, x, reduction="sum") / len(x)
        kl = 0.5 * (mu.pow(2) + logvar.exp() - logvar - 1).sum(1).mean()
        return rec + beta * kl, rec, kl

    vae = VAE().to(device); opt = torch.optim.Adam(vae.parameters(), lr=2e-3)
    Xd = X.to(device)
    for ep in range(300):
        perm = torch.randperm(len(Xd))
        for i in range(0, len(Xd), 128):
            xb = Xd[perm[i:i+128]]; opt.zero_grad()
            x_hat, mu, logvar = vae(xb)
            loss, rec, kl = elbo_loss(x_hat, xb, mu, logvar); loss.backward(); opt.step()
        if ep % 100 == 0 or ep == 299: print(f"época {ep:3d}  -ELBO={loss.item():.2f}  rec={rec.item():.2f}  KL={kl.item():.2f}")

    vae.eval()
    with torch.no_grad():
        mu, _ = vae.encode(Xd)
        g = torch.linspace(-2.5, 2.5, 12); grid = torch.stack(torch.meshgrid(g, g, indexing="xy"), -1).view(-1, 2).to(device)
        imgs = vae.dec(grid).view(12, 12, 8, 8).cpu()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    axes[0].scatter(*mu.cpu().T, c=y, cmap="tab10", s=6); axes[0].set_title("espacio latente 2D del VAE (≈ N(0,I))")
    axes[1].imshow(imgs.permute(0, 2, 1, 3).reshape(96, 96), cmap="gray"); axes[1].axis("off"); axes[1].set_title("decodificando una rejilla del espacio latente"); plt.show()

> **Idea clave para lo que viene:** los generadores de imágenes modernos (Stable Diffusion, FLUX) trabajan en el **espacio latente de un VAE** ("latent diffusion"): comprimen la imagen 8× por lado con un VAE convolucional y hacen difusión/flow matching ahí, lo que abarata el cómputo ~64×.

## 3. Redes generativas antagónicas (GAN) (Géron 17.8; D2L Cap. 20)

Dos redes compiten: el **generador** $G(\mathbf{z})$ intenta producir muestras realistas; el **discriminador** $D(\mathbf{x})$ intenta distinguir reales de falsas:
$$\min_G\max_D\ \mathbb{E}_{x\sim p_{data}}[\log D(x)] + \mathbb{E}_{z}[\log(1-D(G(z)))].$$
En la práctica $G$ maximiza $\log D(G(z))$ (*non-saturating loss*). Problemas típicos: **colapso de modo** (G produce pocas variantes), oscilaciones, sensibilidad a hiperparámetros. Mejoras: DCGAN, WGAN-GP, StyleGAN. Hoy las GAN sobreviven en nichos (superresolución, decodificadores rápidos) pero la difusión/flow matching las desplazó por estabilidad y cobertura de modos.

Entrenamos una GAN sobre una mezcla de 8 gaussianas en 2D: el banco de pruebas clásico para ver el colapso de modo.

In [ ]:
if TORCH:
    def ocho_gaussianas(n):
        ang = rng.integers(0, 8, n) * 2 * np.pi / 8
        return torch.tensor(np.c_[np.cos(ang), np.sin(ang)] * 2 + rng.normal(0, 0.08, (n, 2)), dtype=torch.float32)
    real = ocho_gaussianas(4000).to(device)

    G = nn.Sequential(nn.Linear(2, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2)).to(device)
    D = nn.Sequential(nn.Linear(2, 128), nn.LeakyReLU(0.2), nn.Linear(128, 128), nn.LeakyReLU(0.2), nn.Linear(128, 1)).to(device)
    oG = torch.optim.Adam(G.parameters(), 2e-4, betas=(0.5, 0.999)); oD = torch.optim.Adam(D.parameters(), 2e-4, betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss(); snaps = {}
    for step in range(3001):
        xb = real[torch.randint(0, len(real), (256,))]; z = torch.randn(256, 2, device=device)
        # paso del discriminador
        oD.zero_grad(); fake = G(z).detach()
        lossD = bce(D(xb), torch.ones(256, 1, device=device)) + bce(D(fake), torch.zeros(256, 1, device=device)); lossD.backward(); oD.step()
        # paso del generador (non-saturating)
        oG.zero_grad(); lossG = bce(D(G(z)), torch.ones(256, 1, device=device)); lossG.backward(); oG.step()
        if step in (300, 1000, 3000): snaps[step] = G(torch.randn(2000, 2, device=device)).detach().cpu().numpy()
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].scatter(*real.cpu().T, s=3); axes[0].set_title("reales: 8 modos")
    for ax, (s, pts) in zip(axes[1:], snaps.items()):
        ax.scatter(*pts.T, s=3, color="C1"); ax.set_title(f"GAN paso {s}")
    for ax in axes: ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    plt.suptitle("¿Cubre todos los modos o colapsa? Vuelve a ejecutar con otra semilla y compara"); plt.show()

## 4. Modelos de difusión (DDPM) desde cero (Géron 17.9)

**Proceso hacia adelante** (fijo): añadir ruido gaussiano en $T$ pasos con varianzas $\beta_t$. Con $\alpha_t=1-\beta_t$ y $\bar\alpha_t=\prod_{s\le t}\alpha_s$ se puede saltar directamente a cualquier $t$:
$$\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\,\boldsymbol\epsilon,\qquad \boldsymbol\epsilon\sim\mathcal N(0,I).$$

**Entrenamiento** (Ho et al., 2020): una red $\epsilon_\theta(\mathbf{x}_t, t)$ aprende a predecir el ruido añadido:
$$\mathcal{L} = \mathbb{E}_{t,\mathbf{x}_0,\boldsymbol\epsilon}\big\|\boldsymbol\epsilon - \epsilon_\theta(\mathbf{x}_t,t)\big\|^2 .$$
Esto equivale a aprender el **score** $\nabla_x\log p_t(x) = -\epsilon/\sqrt{1-\bar\alpha_t}$ (Song & Ermon).

**Muestreo**: partir de $\mathbf{x}_T\sim\mathcal N(0,I)$ y deshacer paso a paso
$$\mathbf{x}_{t-1} = \frac{1}{\sqrt{\alpha_t}}\Big(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\epsilon_\theta(\mathbf{x}_t,t)\Big) + \sigma_t\mathbf{z}.$$
El costo: cientos de evaluaciones de la red por muestra (DDIM, *distillation* y *consistency models* lo reducen).

In [ ]:
if TORCH:
    from sklearn.datasets import make_swiss_roll
    sr, _ = make_swiss_roll(6000, noise=0.4, random_state=0); data = torch.tensor(sr[:, [0, 2]] / 6.0, dtype=torch.float32).to(device)

    class TimeMLP(nn.Module):
        '''Red pequeña que recibe (x, t). t se codifica con senos/cosenos (como en Transformers).'''
        def __init__(self, d=2, h=128, n_freq=16):
            super().__init__(); self.freqs = torch.exp(torch.linspace(0, math.log(1000), n_freq)).to(device)
            self.net = nn.Sequential(nn.Linear(d + 2 * n_freq, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, d))
        def forward(self, x, t):                       # t en [0,1]
            ang = t[:, None] * self.freqs[None]; emb = torch.cat([ang.sin(), ang.cos()], 1)
            return self.net(torch.cat([x, emb], 1))

    T = 200
    betas = torch.linspace(1e-4, 0.05, T, device=device); alphas = 1 - betas; abar = torch.cumprod(alphas, 0)

    eps_model = TimeMLP().to(device); opt = torch.optim.Adam(eps_model.parameters(), 1e-3)
    t0 = time.time()
    for step in range(4000):
        x0 = data[torch.randint(0, len(data), (512,))]; t = torch.randint(0, T, (512,), device=device); eps = torch.randn_like(x0)
        xt = abar[t].sqrt()[:, None] * x0 + (1 - abar[t]).sqrt()[:, None] * eps
        loss = F.mse_loss(eps_model(xt, t.float() / T), eps); opt.zero_grad(); loss.backward(); opt.step()
        if step % 1000 == 0: print(f"paso {step}  loss={loss.item():.4f}")
    print(f"entrenado en {time.time()-t0:.1f}s")

    @torch.no_grad()
    def muestrear_ddpm(n=2000):
        x = torch.randn(n, 2, device=device); traza = [x.cpu().numpy()]
        for t in reversed(range(T)):
            eps = eps_model(x, torch.full((n,), t / T, device=device))
            x = (x - betas[t] / (1 - abar[t]).sqrt() * eps) / alphas[t].sqrt()
            if t > 0: x = x + betas[t].sqrt() * torch.randn_like(x)
            if t in (150, 75, 0): traza.append(x.cpu().numpy())
        return traza
    traza = muestrear_ddpm()
    fig, axes = plt.subplots(1, 5, figsize=(19, 3.8))
    axes[0].scatter(*data.cpu().T, s=2); axes[0].set_title("datos (swiss roll)")
    for ax, pts, nombre in zip(axes[1:], traza, ["x_T ~ N(0,I)", "t=150", "t=75", "t=0 (muestras)"]):
        ax.scatter(*pts.T, s=2, color="C3"); ax.set_title(nombre)
    for ax in axes: ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    plt.suptitle(f"DDPM: {T} pasos de eliminación de ruido"); plt.show()

## 5. Flow Matching: el estado del arte 2024-2026

En vez de un proceso estocástico con cientos de pasos, **Flow Matching** (Lipman et al., 2023; *Rectified Flow*, Liu et al., 2023) aprende directamente un **campo de velocidad** $v_\theta(\mathbf{x},t)$ que transporta ruido a datos a lo largo de una **EDO** (¡Módulo 1!):
$$\frac{d\mathbf{x}}{dt} = v_\theta(\mathbf{x}, t),\qquad \mathbf{x}(0)\sim\mathcal N(0,I),\ \mathbf{x}(1)\sim p_{data}.$$

Con la **interpolación lineal** $\mathbf{x}_t = (1-t)\,\mathbf{x}_0 + t\,\mathbf{x}_1$ (ruido $\mathbf{x}_0$, dato $\mathbf{x}_1$) la velocidad objetivo es simplemente $\mathbf{x}_1-\mathbf{x}_0$, y la pérdida:
$$\mathcal{L}_{FM} = \mathbb{E}_{t,\mathbf{x}_0,\mathbf{x}_1}\big\|v_\theta(\mathbf{x}_t,t) - (\mathbf{x}_1-\mathbf{x}_0)\big\|^2 .$$

Ventajas: sin *schedule* de ruido, trayectorias casi rectas → **muestreo con 5-50 pasos de Euler**, entrenamiento más estable. Es el objetivo de entrenamiento de **Stable Diffusion 3, FLUX.1, Meta Movie Gen, y muchos modelos de audio/vídeo** actuales. Comparamos con el DDPM anterior en calidad vs número de pasos.

In [ ]:
if TORCH:
    v_model = TimeMLP().to(device); opt = torch.optim.Adam(v_model.parameters(), 1e-3)
    t0 = time.time()
    for step in range(4000):
        x1 = data[torch.randint(0, len(data), (512,))]; x0 = torch.randn_like(x1); t = torch.rand(512, device=device)
        xt = (1 - t)[:, None] * x0 + t[:, None] * x1
        loss = F.mse_loss(v_model(xt, t), x1 - x0); opt.zero_grad(); loss.backward(); opt.step()
        if step % 1000 == 0: print(f"paso {step}  loss={loss.item():.4f}")
    print(f"entrenado en {time.time()-t0:.1f}s")

    @torch.no_grad()
    def muestrear_fm(n=2000, pasos=10):
        x = torch.randn(n, 2, device=device); h = 1.0 / pasos
        for i in range(pasos):                                # integrador de Euler
            x = x + h * v_model(x, torch.full((n,), i * h, device=device))
        return x.cpu().numpy()

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, pasos in zip(axes, [1, 4, 10, 50]):
        ax.scatter(*muestrear_fm(pasos=pasos).T, s=2, color="C2"); ax.set_title(f"Flow Matching, {pasos} paso(s) de Euler"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    plt.suptitle("Con ~10 pasos ya reproduce la estructura; DDPM necesitó 200"); plt.show()

In [ ]:
if TORCH:
    # Métrica cuantitativa sencilla: distancia energética (energy distance) entre muestras y datos, por número de pasos
    def energy_distance(a, b, n=1500):
        a = torch.tensor(a[:n]).to(device); b = b[:n]
        ab = torch.cdist(a, b).mean(); aa = torch.cdist(a, a).mean(); bb = torch.cdist(b, b).mean()
        return (2 * ab - aa - bb).item()
    for pasos in [2, 5, 10, 25, 50]:
        print(f"Flow Matching {pasos:3d} pasos -> ED = {energy_distance(muestrear_fm(pasos=pasos), data):.4f}")
    print(f"DDPM         {T:3d} pasos -> ED = {energy_distance(traza[-1], data):.4f}")

### Guía de decisión rápida (2026)
| Necesidad | Técnica |
|---|---|
| Representación comprimida / detección de anomalías | Autoencoder (o VAE) |
| Generación con espacio latente interpretable, datos pequeños | VAE, β-VAE |
| Máxima calidad de imagen/vídeo/audio, control por texto | **Flow matching / difusión latente** con *backbone* Transformer (DiT/MMDiT) |
| Latencia muy baja (1 paso) | Distilación / *consistency models* / GAN |
| Datos tabulares sintéticos | TabDDPM, CTGAN, o modelos de lenguaje tabulares |

## 6. Aprendizaje auto-supervisado (SSL)

Sin etiquetas, definimos una **tarea pretexto** cuya solución requiera entender los datos:
* **Contrastivo (SimCLR, 2020):** dos *aumentos* de la misma imagen deben tener *embeddings* cercanos; los de imágenes distintas, lejanos. Pérdida **InfoNCE / NT-Xent**:
$$\ell_{i,j} = -\log\frac{\exp(\mathrm{sim}(z_i,z_j)/\tau)}{\sum_{k\ne i}\exp(\mathrm{sim}(z_i,z_k)/\tau)}.$$
* **Auto-destilación (BYOL, DINO/DINOv2, 2021-2023):** red estudiante imita a una red profesora (media móvil), sin negativos. **DINOv2** es hoy el *backbone* visual "universal" por defecto.
* **Enmascarado (MAE, 2022; BERT en texto):** ocultar el 75 % de los parches y reconstruirlos.

Implementamos la pérdida NT-Xent y un SimCLR minimalista sobre `digits` con aumentos simples (ruido + desplazamiento), y comprobamos que las representaciones aprendidas **sin etiquetas** separan las clases mejor que los píxeles crudos (evaluación por *linear probe*/KNN, el protocolo estándar).

In [ ]:
if TORCH:
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.model_selection import train_test_split

    def nt_xent(z1, z2, tau=0.2):
        z = F.normalize(torch.cat([z1, z2]), dim=1); n = len(z1)
        sim = z @ z.T / tau; sim.fill_diagonal_(-1e9)                 # excluir auto-similitud
        targets = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)   # el positivo de i es i+n
        return F.cross_entropy(sim, targets)

    def aumentar(x):                                                 # x: (B,64) imágenes 8x8 en [0,1]
        img = x.view(-1, 1, 8, 8)
        dx, dy = np.random.randint(-1, 2, 2); img = torch.roll(img, shifts=(int(dx), int(dy)), dims=(2, 3))
        img = img + 0.15 * torch.randn_like(img)
        mask = (torch.rand(len(img), 1, 8, 8, device=img.device) > 0.15).float()      # 'cutout' aleatorio
        return (img * mask).clamp(0, 1).view(-1, 64)

    encoder = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 128)).to(device)
    proj = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 32)).to(device)
    opt = torch.optim.AdamW(list(encoder.parameters()) + list(proj.parameters()), 1e-3)
    for step in range(1500):
        xb = Xd[torch.randint(0, len(Xd), (256,))]
        loss = nt_xent(proj(encoder(aumentar(xb))), proj(encoder(aumentar(xb)))); opt.zero_grad(); loss.backward(); opt.step()
        if step % 500 == 0: print(f"paso {step}  NT-Xent={loss.item():.3f}")

    with torch.no_grad(): H = encoder(Xd).cpu().numpy()
    Xtr, Xte, ytr, yte, Htr, Hte = train_test_split(X.numpy(), y, H, test_size=0.5, stratify=y, random_state=0)
    # protocolo: pocas etiquetas (50 por... 5%) para ver el valor de la representación
    few = np.random.default_rng(0).random(len(ytr)) < 0.05
    for nombre, A, B in [("píxeles crudos", Xtr, Xte), ("representación SimCLR", Htr, Hte)]:
        knn = KNeighborsClassifier(5).fit(A[few], ytr[few]); print(f"KNN con {few.sum()} etiquetas sobre {nombre:22s}: {knn.score(B, yte):.3f}")

## 7. Ecosistema práctico: Hugging Face `diffusers`, LoRA y nube (Fregly Cap. 10-11)

```python
# OPCIONAL (requiere GPU y descarga de pesos)
from diffusers import FluxPipeline            # o StableDiffusion3Pipeline, AutoPipelineForText2Image
pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16).to("cuda")
img = pipe("a watercolor of a Yu-Gi-Oh! card about gradient descent", num_inference_steps=4).images[0]
```
* **Personalización:** LoRA / DreamBooth sobre el *backbone* (ver LoRA en el Módulo 6); ControlNet e IP-Adapter para control espacial y de estilo.
* **Producción:** cuantización (fp8/int8), *offloading*, `torch.compile`; en AWS: SageMaker + Bedrock (Fregly Cap. 12), en GCP Vertex AI, en Azure AI Foundry.
* **Evaluación:** FID/KID ya no bastan; se usan métricas basadas en CLIP/DINO y evaluación humana o con modelos juez (Módulo 6).
* **Legal/ética:** procedencia de datos de entrenamiento, marcas de agua (C2PA), obligaciones de transparencia del **AI Act** para contenido sintético (Art. 50, aplicable desde agosto 2026 — Módulo 10).

## 📦 Recursos de los repositorios oficiales de los libros

| Libro | Recurso | Para qué usarlo aquí |
|---|---|---|
| Géron, *Hands-On ML 3* | [`17_autoencoders_gans_and_diffusion_models.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/17_autoencoders_gans_and_diffusion_models.ipynb) | Autoencoders apilados/convolucionales/*denoising*/dispersos, VAE sobre Fashion-MNIST, GAN y DCGAN, y un **DDPM completo sobre imágenes** en Keras: el paso natural tras nuestro DDPM 2D |
| Zhang et al., *D2L* | [Cap. 20 GANs](https://d2l.ai/chapter_generative-adversarial-networks/index.html) (GAN básica y DCGAN sobre Pokémon) | Implementación PyTorch de referencia de la sección 3 |
| Fregly et al., *Generative AI on AWS* | [`10_multimodal/`](https://github.com/generative-ai-on-aws/generative-ai-on-aws/tree/main/10_multimodal), [`11_diffusers/`](https://github.com/generative-ai-on-aws/generative-ai-on-aws/tree/main/11_diffusers) (Stable Diffusion con `diffusers`: text-to-image, ControlNet, DreamBooth/LoRA), [`04_optimize/`](https://github.com/generative-ai-on-aws/generative-ai-on-aws/tree/main/04_optimize) (cuantización, FSDP) | Los notebooks de `11_diffusers` son el "cómo se hace en producción" de la sección 7; requieren GPU (SageMaker o Colab) |
| Alammar & Grootendorst (bonus) | [*The Illustrated Stable Diffusion*](https://jalammar.github.io/illustrated-stable-diffusion/) | Explicación visual del pipeline de difusión latente |

## 8. Ejercicios
1. **β-VAE.** Entrena el VAE con `beta` ∈ {0.1, 1, 4}. ¿Cómo cambian la reconstrucción y la "organización" del espacio latente? Relaciónalo con el *disentanglement*.
2. **Denoising AE.** Añade ruido gaussiano a la entrada del AE lineal y no lineal; mide la reconstrucción del dígito limpio. ¿Por qué esto está relacionado con la difusión?
3. **Colapso de modo.** Repite la GAN con `lr=1e-3` y sin `betas=(0.5, 0.999)`. Cuenta cuántos modos cubre (asigna cada muestra al centro más cercano). Implementa WGAN-GP y compara.
4. **DDIM.** Implementa el muestreo determinista de DDIM (Song et al., 2021) con 20 pasos sobre el mismo `eps_model`. Compara la distancia energética con DDPM-200 y FM-10.
5. **Flow matching condicional.** Añade la etiqueta de clase (0-9) como entrada a `v_model` y entrena sobre `digits` (64-D). Genera dígitos de una clase pedida. Implementa *classifier-free guidance* (entrena con la etiqueta descartada el 10 % de las veces).
6. **SSL vs supervisado.** Con el mismo `encoder`, entrena una cabeza supervisada con el 100 % de etiquetas. ¿Cuánta ventaja mantiene SimCLR cuando hay 1 %, 5 %, 20 % de etiquetas?

## Referencias
* Géron, A. (2022). *Hands-On ML* (3.ª ed.). Cap. 17.
* Zhang, A. et al. *Dive into Deep Learning*. Cap. 20.
* Fregly, C., Barth, A. & Eigenbrode, S. (2023). *Generative AI on AWS*. O'Reilly. Cap. 4, 10-12. Código: github.com/generative-ai-on-aws/generative-ai-on-aws
* Kingma, D. P. & Welling, M. (2014). *Auto-Encoding Variational Bayes*. ICLR.
* Goodfellow, I. et al. (2014). *Generative Adversarial Nets*. NeurIPS.
* Ho, J., Jain, A. & Abbeel, P. (2020). *Denoising Diffusion Probabilistic Models*. NeurIPS.
* Rombach, R. et al. (2022). *High-Resolution Image Synthesis with Latent Diffusion Models*. CVPR.
* Lipman, Y. et al. (2023). *Flow Matching for Generative Modeling*. ICLR. · Liu, X. et al. (2023). *Flow Straight and Fast: Rectified Flow*. ICLR.
* Esser, P. et al. (2024). *Scaling Rectified Flow Transformers for High-Resolution Image Synthesis* (Stable Diffusion 3). ICML.
* Holderrieth, P. & Erives, E. (2025). *An Introduction to Flow Matching and Diffusion Models* (MIT 6.S184). arXiv:2506.02070.
* Chen, T. et al. (2020). *SimCLR*. ICML. · Oquab, M. et al. (2024). *DINOv2*. TMLR. · He, K. et al. (2022). *Masked Autoencoders*. CVPR.